# E-Commerce Business Analytics Engine

## Overview
This project builds a SQL-driven business analytics engine on top of a PostgreSQL database simulating 2 years of e-commerce operations across 10,000 customers, 200 products, and 50,000 orders.

**Business Questions Answered:**
- How has monthly revenue trended over time?
- Which product categories drive the most revenue and profit?
- What percentage of revenue comes from the top 10% of customers?
- How are customers segmented by value?
- Which cohorts of customers retain best over time?
- What is the repeat purchase rate?

**Tech Stack:**
- PostgreSQL — database design, querying, window functions, CTEs
- Python — data generation, visualization
- pandas, matplotlib, psycopg2

## Setup: Connect to Database

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conn = psycopg2.connect(
    dbname="business_analytics_engine",
    user="postgres",
    password="vaishvee",
    host="localhost",
    port="5432"
)

print("Connected to database successfully")

## 1. Dataset Overview

Before analysis, let's understand the scale of the data we're working with.

In [ ]:
# Dataset scale
queries = {
    "Total Customers": "SELECT COUNT(*) FROM customers",
    "Total Products": "SELECT COUNT(*) FROM products",
    "Total Orders": "SELECT COUNT(*) FROM orders",
    "Completed Orders": "SELECT COUNT(*) FROM orders WHERE status = 'completed'",
    "Total Order Items": "SELECT COUNT(*) FROM order_items",
    "Total Returns": "SELECT COUNT(*) FROM returns"
}

for label, query in queries.items():
    result = pd.read_sql(query, conn).iloc[0, 0]
    print(f"{label}: {result:,}")

## 2. Monthly Revenue Trend

Tracking revenue over time is the most fundamental business health metric. A growing trend indicates business expansion while dips signal potential issues worth investigating.

In [ ]:
monthly_revenue = pd.read_sql("""
    SELECT 
        DATE_TRUNC('month', o.order_date) AS month,
        ROUND(SUM(oi.quantity * oi.selling_price), 2) AS monthly_revenue,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY DATE_TRUNC('month', o.order_date)
    ORDER BY month
""", conn)

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(monthly_revenue["month"], monthly_revenue["monthly_revenue"],
         marker="o", color="steelblue", linewidth=2, markersize=4, label="Revenue")
ax1.set_ylabel("Monthly Revenue", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.bar(monthly_revenue["month"], monthly_revenue["total_orders"],
        alpha=0.3, color="darkorange", width=20, label="Orders")
ax2.set_ylabel("Total Orders", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")

ax1.set_title("Monthly Revenue and Order Volume", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Revenue and Profit by Category

Understanding which product categories drive the most revenue AND profit helps prioritize inventory, marketing spend, and supplier negotiations.

In [ ]:
category_revenue = pd.read_sql("""
    SELECT 
        p.category,
        ROUND(SUM(oi.quantity * oi.selling_price), 2) AS category_revenue,
        ROUND(SUM(oi.quantity * p.cost), 2) AS total_cost,
        ROUND(SUM(oi.quantity * (oi.selling_price - p.cost)), 2) AS category_profit,
        ROUND(
            SUM(oi.quantity * (oi.selling_price - p.cost)) * 100.0
            / SUM(oi.quantity * oi.selling_price),
        2) AS profit_margin_percent
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY p.category
    ORDER BY category_revenue DESC
""", conn)

print(category_revenue.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue vs Profit
x = range(len(category_revenue))
width = 0.35
axes[0].bar([i - width/2 for i in x], category_revenue["category_revenue"],
            width, label="Revenue", color="steelblue", edgecolor="white")
axes[0].bar([i + width/2 for i in x], category_revenue["category_profit"],
            width, label="Profit", color="darkorange", edgecolor="white")
axes[0].set_title("Revenue vs Profit by Category", fontsize=12, fontweight="bold")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(category_revenue["category"])
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Profit Margin
axes[1].bar(category_revenue["category"], category_revenue["profit_margin_percent"],
            color="steelblue", edgecolor="white")
for i, val in enumerate(category_revenue["profit_margin_percent"]):
    axes[1].text(i, val + 0.2, f"{val}%", ha="center", fontsize=9)
axes[1].set_title("Profit Margin % by Category", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Profit Margin (%)")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Category Performance Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Customer Revenue Analysis

Understanding how revenue is distributed across customers reveals concentration risk. If too much revenue comes from too few customers, the business is vulnerable to churn of key accounts.

In [ ]:
# Top 10% revenue concentration
top10 = pd.read_sql("""
    WITH customer_revenue AS (
        SELECT 
            o.customer_id,
            SUM(oi.quantity * oi.selling_price) AS revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY o.customer_id
    ),
    ranked_customers AS (
        SELECT *,
            NTILE(10) OVER (ORDER BY revenue DESC) AS decile
        FROM customer_revenue
    )
    SELECT 
        ROUND(
            SUM(revenue) FILTER (WHERE decile = 1) * 100.0
            / SUM(revenue),
        2) AS top_10_percent_revenue_share
    FROM ranked_customers
""", conn)

print(f"Top 10% of customers generate {top10.iloc[0,0]}% of total revenue")

# Revenue distribution
customer_revenue = pd.read_sql("""
    SELECT 
        o.customer_id,
        ROUND(SUM(oi.quantity * oi.selling_price), 2) AS customer_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id
    ORDER BY customer_revenue DESC
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(customer_revenue["customer_revenue"], bins=50,
        color="steelblue", edgecolor="white")
ax.set_title("Customer Revenue Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Revenue per Customer")
ax.set_ylabel("Number of Customers")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Customer Segmentation

Segmenting customers by purchase frequency and monetary value helps identify which customers to retain, which to upsell, and which are at risk of churning.

In [ ]:
segments = pd.read_sql("""
    WITH rfm AS (
        SELECT 
            o.customer_id,
            COUNT(DISTINCT o.order_id) AS frequency,
            ROUND(SUM(oi.quantity * oi.selling_price), 2) AS monetary
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY o.customer_id
    )
    SELECT 
        CASE 
            WHEN frequency >= 5 AND monetary >= 5000 THEN 'High Value'
            WHEN frequency >= 3 THEN 'Mid Value'
            ELSE 'Low Value'
        END AS customer_segment,
        COUNT(*) AS customer_count,
        ROUND(AVG(monetary), 2) AS avg_revenue,
        ROUND(AVG(frequency), 2) AS avg_orders
    FROM rfm
    GROUP BY customer_segment
    ORDER BY avg_revenue DESC
""", conn)

print(segments.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["steelblue", "darkorange", "green"]
axes[0].pie(segments["customer_count"], labels=segments["customer_segment"],
            autopct="%1.1f%%", colors=colors, startangle=140,
            wedgeprops={"edgecolor": "white"})
axes[0].set_title("Customer Count by Segment", fontsize=12, fontweight="bold")

axes[1].bar(segments["customer_segment"], segments["avg_revenue"],
            color=colors, edgecolor="white")
for i, val in enumerate(segments["avg_revenue"]):
    axes[1].text(i, val + 50, f"{val:,.0f}", ha="center", fontsize=9)
axes[1].set_title("Average Revenue by Segment", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Average Revenue")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Customer Segmentation Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Cohort Retention Analysis

Cohort analysis tracks what percentage of customers acquired in a given month are still purchasing in subsequent months. This is one of the most powerful metrics for understanding long-term business health.

In [ ]:
cohort = pd.read_sql("""
    WITH customer_cohort AS (
        SELECT 
            customer_id,
            DATE_TRUNC('month', MIN(order_date)) AS cohort_month
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),
    customer_activity AS (
        SELECT 
            o.customer_id,
            DATE_TRUNC('month', o.order_date) AS activity_month
        FROM orders o
        WHERE o.status = 'completed'
        GROUP BY o.customer_id, DATE_TRUNC('month', o.order_date)
    ),
    cohort_data AS (
        SELECT 
            c.cohort_month,
            a.activity_month,
            EXTRACT(YEAR FROM AGE(a.activity_month, c.cohort_month)) * 12 +
            EXTRACT(MONTH FROM AGE(a.activity_month, c.cohort_month)) AS months_since_acquisition,
            COUNT(DISTINCT a.customer_id) AS active_customers
        FROM customer_cohort c
        JOIN customer_activity a ON c.customer_id = a.customer_id
        GROUP BY c.cohort_month, a.activity_month
    ),
    cohort_sizes AS (
        SELECT 
            cohort_month,
            COUNT(DISTINCT customer_id) AS cohort_size
        FROM customer_cohort
        GROUP BY cohort_month
    )
    SELECT 
        cd.cohort_month,
        cs.cohort_size,
        cd.months_since_acquisition,
        cd.active_customers,
        ROUND(cd.active_customers * 100.0 / cs.cohort_size, 2) AS retention_rate_percent
    FROM cohort_data cd
    JOIN cohort_sizes cs ON cd.cohort_month = cs.cohort_month
    WHERE cd.months_since_acquisition <= 12
    ORDER BY cd.cohort_month, cd.months_since_acquisition
""", conn)

# Pivot for heatmap
cohort_pivot = cohort.pivot_table(
    index="cohort_month",
    columns="months_since_acquisition",
    values="retention_rate_percent"
)

cohort_pivot.index = cohort_pivot.index.strftime("%Y-%m")

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(cohort_pivot.values, cmap="Blues", aspect="auto")

ax.set_xticks(range(len(cohort_pivot.columns)))
ax.set_xticklabels([f"Month {int(c)}" for c in cohort_pivot.columns])
ax.set_yticks(range(len(cohort_pivot.index)))
ax.set_yticklabels(cohort_pivot.index)

for i in range(len(cohort_pivot.index)):
    for j in range(len(cohort_pivot.columns)):
        val = cohort_pivot.values[i, j]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                    fontsize=7, color="white" if val > 50 else "black")

plt.colorbar(im, ax=ax, label="Retention Rate %")
ax.set_title("Customer Cohort Retention Heatmap", fontsize=14, fontweight="bold")
ax.set_xlabel("Months Since Acquisition")
ax.set_ylabel("Cohort Month")
plt.tight_layout()
plt.show()

## 7. Repeat Purchase Rate and Payment Analysis

In [ ]:
# Repeat purchase rate
repeat_rate = pd.read_sql("""
    WITH customer_orders AS (
        SELECT 
            customer_id,
            COUNT(DISTINCT order_id) AS total_orders
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    )
    SELECT 
        ROUND(
            COUNT(*) FILTER (WHERE total_orders > 1) * 100.0
            / COUNT(*),
        2) AS repeat_purchase_rate_percent
    FROM customer_orders
""", conn)

print(f"Repeat Purchase Rate: {repeat_rate.iloc[0,0]}%")

# Payment method distribution
payments = pd.read_sql("""
    SELECT 
        payment_method,
        COUNT(*) AS total_transactions,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(COUNT(*) FILTER (WHERE payment_status = 'failed') * 100.0 / COUNT(*), 2) AS failure_rate
    FROM payments
    GROUP BY payment_method
    ORDER BY total_transactions DESC
""", conn)

print("\nPayment Method Analysis:")
print(payments.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(payments["total_transactions"], labels=payments["payment_method"],
            autopct="%1.1f%%", startangle=140,
            colors=["steelblue", "darkorange", "green", "crimson"],
            wedgeprops={"edgecolor": "white"})
axes[0].set_title("Payment Method Distribution", fontsize=12, fontweight="bold")

axes[1].bar(payments["payment_method"], payments["failure_rate"],
            color="crimson", edgecolor="white")
for i, val in enumerate(payments["failure_rate"]):
    axes[1].text(i, val + 0.2, f"{val}%", ha="center", fontsize=9)
axes[1].set_title("Payment Failure Rate by Method", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Failure Rate (%)")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Payment Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Key Business Insights

**1. Revenue is evenly distributed across categories**
No single category dominates revenue, suggesting a well-diversified product portfolio. However, profit margins vary significantly across categories — this should drive pricing and procurement decisions.

**2. Top 10% of customers drive disproportionate revenue**
A small segment of high-frequency, high-spend customers generates a significant share of total revenue. Retaining these customers through loyalty programs should be a top priority.

**3. Cohort retention drops sharply after Month 1**
Most cohorts show strong initial activity but significant drop-off after the first month. This signals a need for stronger post-purchase engagement — email campaigns, recommendations, or loyalty incentives.

**4. Repeat purchase rate reveals loyalty baseline**
The repeat purchase rate establishes a baseline for measuring the impact of retention initiatives over time.

**5. Payment failure rates vary by method**
Some payment methods show higher failure rates than others. Prioritizing lower-failure methods in the checkout flow could directly improve conversion rates.

## 9. Limitations and What I Would Do Next

**Current Limitations:**
- Data is synthetically generated — real e-commerce data would have seasonality, promotional effects, and external factors
- RFM segmentation uses fixed thresholds — a clustering approach (K-means) would produce more data-driven segments
- Cohort analysis only tracks purchase activity — a true retention metric would track active engagement

**What I Would Build Next:**
- Customer churn prediction model using purchase history features
- Price elasticity analysis per category
- A/B test framework for promotional campaign evaluation
- Interactive Streamlit dashboard for non-technical stakeholders
- Connect to real dataset from Kaggle (e.g. Brazilian E-Commerce dataset by Olist)